# 本地 LLM Vision（OpenAI 兼容，例如 LM Studio）

- 服务根地址：`http://127.0.0.1:1234`；Python 里请把 **base_url** 设为 `http://127.0.0.1:1234/v1`（多出来的 `/v1` 是 OpenAI 兼容路由）。
- 在 LM Studio 中需**加载支持视觉的多模态模型**；若未设置环境变量 `LOCAL_VISION_MODEL`，脚本会自动选用当前服务上的第一个模型。
- 示例用一张极小的内联 PNG（base64），不依赖本地图片文件。

In [ ]:
%pip install -q openai

In [ ]:
import os
from openai import OpenAI

# 服务根地址一般是 http://127.0.0.1:1234 ，OpenAI 兼容接口需在路径上加 /v1
BASE_URL = os.environ.get("OPENAI_BASE_URL", "http://127.0.0.1:1234/v1")
API_KEY = os.environ.get("OPENAI_API_KEY", "lm-studio")
MODEL = os.environ.get("LOCAL_VISION_MODEL", "replace-with-your-model-id")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# 未配置 LOCAL_VISION_MODEL 时，自动用当前服务上的第一个模型（LM Studio 通常只加载一个）
if MODEL == "replace-with-your-model-id":
    models = client.models.list().data
    if not models:
        raise RuntimeError("本地服务未返回任何模型，请先在 LM Studio 等中加载模型。")
    MODEL = models[0].id
    print("使用模型:", MODEL)

In [ ]:
# 可选：列出当前服务上的模型 id，把上一单元里的 MODEL 改成其中之一（须为支持 vision 的模型）
for m in client.models.list().data:
    print(m.id)

In [ ]:
# 1x1 PNG（base64 data URL，无需本地文件）
TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg=="
)
image_url = f"data:image/png;base64,{TINY_PNG_B64}"

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "请用一句话描述这张图片（若看不清则说明即可）。"},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }
    ],
    max_tokens=256,
)

print(resp.choices[0].message.content)